<a href="https://colab.research.google.com/github/ataakif1901/tgb/blob/master/rvc-webui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! nvidia-smi
! nvcc -V
! free -h

Sun Sep 27 21:28:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os

# @markdown # Mount Google Drive
data_path = "/content/data"
mount_gdrive = True  # @param {type:"boolean"}
gdrive_data_path = "/content/drive/MyDrive/AI/rvc-webui"  # @param {type:"string"}

if mount_gdrive:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(gdrive_data_path, exist_ok=True)
    ! ln -s {data_path} {gdrive_data_path}
else:
    os.makedirs(data_path, exist_ok=True)

Mounted at /content/drive


In [3]:
# @markdown # Clone repository

%cd /content/
repository_url = "https://github.com/ddPn08/rvc-webui.git"  # @param {type: "string"}
webui_branch = "main"  # @param {type: "string"}

! git clone {repository_url}
%cd /content/rvc-webui

/content
Cloning into 'rvc-webui'...
remote: Enumerating objects: 1135, done.
remote: Counting objects: 100% (524/524), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 1135 (delta 467), reused 435 (delta 435), pack-reused 611 (from 1)
Receiving objects: 100% (1135/1135), 406.49 KiB | 1.66 MiB/s, done.
Resolving deltas: 100% (779/779), done.
/content/rvc-webui


In [4]:
# @markdown # Initialize environment
import os

conda_dir = "/opt/conda"  # @param{type:"string"}
conda_bin = os.path.join(conda_dir, "bin", "conda")

if not os.path.exists(conda_bin):
    ! curl -O https://repo.anaconda.com/miniconda/Miniconda3-py310_23.5.0-3-Linux-x86_64.sh
    ! chmod +x Miniconda3-py310_23.5.0-3-Linux-x86_64.sh
    ! bash ./Miniconda3-py310_23.5.0-3-Linux-x86_64.sh -b -f -p {conda_dir}
    ! rm Miniconda3-py310_23.5.0-3-Linux-x86_64.sh

def run_script(s):
    ! {s}

def make_args(d):
    arguments = ""
    for k, v in d.items():
        if type(v) == bool:
            arguments += f"--{k} " if v else ""
        elif type(v) == str and v:
            arguments += f"--{k} \"{v}\" "
        elif v:
            arguments += f"--{k}={v} "
    return arguments

if os.path.exists("requirments.txt"):
    ! mv requirments.txt requirements.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 91.3M  100 91.3M    0     0   115M      0 --:--:-- --:--:-- --:--:--  115M
PREFIX=/opt/conda
Unpacking payload ...
                                                                                
Installing base environment...





Preparing transaction: - \ | / - done
Executing transaction: | / - \ | / - \ | / - \ | / - \ | / - \ done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /opt/conda


In [6]:
!pip install gradio==3.34.0 gradio_client==0.2.7

INFO: pip is looking at multiple versions of mdit-py-plugins to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of mdit-py-plugins to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.4/288.4 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.5/84.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: markdown-it-py
    Found existing installation: markdown-it-py 4.2.0
    Uninstalling markdown-it-py-4.2.0:


In [18]:
!pip install setuptools==69.5.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 20.5 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.10.2
    Uninstalling setuptools-80.10.2:
      Successfully uninstalled setuptools-80.10.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [17]:
# @markdown # Launch webui
run_script(f"""
eval "$({conda_bin} shell.bash hook)"
python launch.py --share --host 0.0.0.0 --port 41130
""")

Python 3.10.12 (main, Jul  5 2023, 18:54:27) [GCC 11.2.0]
Commit hash: b71742809a24cd89eb18081b831c0b1ac11ccb2a
Installing requirements
Traceback (most recent call last):
  File "/content/rvc-webui/webui.py", line 3, in <module>
    from modules import cmd_opts, ui
  File "/content/rvc-webui/modules/ui.py", line 5, in <module>
    import gradio as gr
  File "/opt/conda/lib/python3.10/site-packages/gradio/__init__.py", line 3, in <module>
    import gradio.components as components
  File "/opt/conda/lib/python3.10/site-packages/gradio/components/__init__.py", line 1, in <module>
    from gradio.components.annotated_image import AnnotatedImage
  File "/opt/conda/lib/python3.10/site-packages/gradio/components/annotated_image.py", line 13, in <module>
    from gradio.components.base import IOComponent, _Keywords
  File "/opt/conda/lib/python3.10/site-packages/gradio/components/base.py", line 29, in <module>
    from gradio.blocks import Block, BlockContext
  File "/opt/conda/lib/python3.10